<a href="https://colab.research.google.com/github/abhishek18-blog/DataScience-and-ML/blob/main/RAG_YouTube_Tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import warnings
warnings.filterwarnings('ignore')

# Install necessary libraries
!pip install langchain_community langchainhub chromadb langchain langchain-openai

In [ ]:
from google.colab import userdata
import os

# Set OpenAI API key from Colab user data
os.environ['OPENAI_API_KEY'] = userdata.get('openAIYtKey')

In [ ]:
from langchain_community.document_loaders import WebBaseLoader

# Load documents from a specified web path
loader = WebBaseLoader(web_paths=["https://www.educosys.com/course/genai"])
docs = loader.load()

# Print the loaded documents
print(docs)

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Initialize a text splitter with a chunk size of 1000 and overlap of 200
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)

# Split the loaded documents into smaller chunks
splits = text_splitter.split_documents(docs)

In [ ]:
# Print the first three splits to inspect their content
print(splits[0])
print(splits[1])
print(splits[2])

In [ ]:
# Print the total number of splits created
print(len(splits))

In [ ]:
from langchain.embeddings.openai import OpenAIEmbeddings
from langchain.vectorstores import Chroma

# Create a Chroma vector store from the document splits using OpenAI embeddings
vectorstore = Chroma.from_documents(documents=splits, embedding=OpenAIEmbeddings())

In [ ]:
# Print the number of documents in the Chroma collection
print(vectorstore._collection.count())

In [ ]:
# Retrieve and print all documents from the Chroma collection (for inspection)
print(vectorstore._collection.get())

In [ ]:
# Retrieve and print specific documents by their IDs, including embeddings and content
print("\nCollection 1 - ", vectorstore._collection.get(ids=['28651d9a-ab51-41f8-ab83-e68285623c4e'], include=["embeddings", "documents"]))
print("\nCollection 2 - ", vectorstore._collection.get(ids=['054dee19-19ed-4574-bc51-511060fd707a'], include=["embeddings", "documents"]))
print("\nCollection 3 - ", vectorstore._collection.get(ids=['2fd71cb4-835a-43c5-b920-b7e1be51c450'], include=["embeddings", "documents"]))

In [ ]:
# Convert the vector store into a retriever for similarity search
retriever = vectorstore.as_retriever()

In [ ]:
from langchain import hub

# Pull a RAG prompt from Langchain Hub
prompt = hub.pull("rlm/rag-prompt")

In [ ]:
from langchain_openai import ChatOpenAI

# Initialize the ChatOpenAI language model
llm = ChatOpenAI()

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [ ]:
def format_docs(docs):
  # Format the retrieved documents into a single string
  return "\n".join(doc.page_content for doc in docs)

In [ ]:
# Construct the RAG chain
rag_chain = ({"context" : retriever | format_docs, "question" : RunnablePassthrough()}
             | prompt
             | llm
             | StrOutputParser())

In [ ]:
# Invoke the RAG chain with a question about course recordings
rag_chain.invoke("Are the recordings of the course available? For how long?")

In [ ]:
# Invoke the RAG chain with a question about course testimonials
rag_chain.invoke("Are the testimonials for the course available? Name the studenst who have shared testimonials")

In [ ]:
# Invoke the RAG chain with a question about course certificates
rag_chain.invoke("Are the certificates for the course provided?")

In [ ]:
# Invoke the RAG chain with a question about projects covered in the course
rag_chain.invoke("What all projects are covered in the course?")

In [ ]:
from langchain_core.runnables import RunnableLambda

In [ ]:
def print_prompt(prompt_text):
  # Helper function to print the prompt text
  print("Prompt - ", prompt_text)
  return prompt_text

In [ ]:
# Construct a RAG chain that also prints the generated prompt
rag_chain_with_print = ({"context" : retriever | format_docs, "question" : RunnablePassthrough()}
             | prompt
             | RunnableLambda(print_prompt)
             | llm
             | StrOutputParser())

In [ ]:
# Invoke the RAG chain with prompt printing for a question about course projects
rag_chain_with_print.invoke("What all projects are covered in the course?")

In [ ]:
# The above chain demonstrates how to inspect the prompt being sent to the LLM.
# This can be useful for debugging and understanding the RAG process.